# Visualize OpenAI Tokenization using [tiktoken](https://github.com/openai/tiktoken)

In [1]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")
assert enc.decode(enc.encode("hello world")) == "hello world"

# To get the tokeniser corresponding to a specific model in the OpenAI API:
encoding = tiktoken.encoding_for_model("gpt-4o")

In [2]:
encoding.encode("tiktoken is great!") # example from: https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb (30.10.2025)

[83, 8251, 2488, 382, 2212, 0]

In [3]:
encoding.decode([83, 8251, 2488, 382, 2212, 0])

'tiktoken is great!'

In [4]:
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [5]:
ENCODING_NAME = "o200k_base"
num_tokens_from_string("tiktoken is great!", encoding_name=ENCODING_NAME)

6

## Number of tokens in a longer text from student essay dataset

In [6]:
from genai_detection.mongo_db.mongo_utils import ParaphraseMongoDB
from genai_detection.config import CONFIG

In [7]:
import os

mongo_db = ParaphraseMongoDB(local_ray=os.path.exists("/Users/klara"))

Successfully connected as MongoDB root user!
Collection 'score' already exists. Skipping creation.


In [8]:
max_docs = 1  # for testing, limit to first 5 documents
INPUT_PRICE_PER_ONE_M_TOKENS_USD = 0.250  # gpt-5 mini pricing as of October 2025
OUTPUT_PRICE_PER_ONE_M_TOKENS_USD = 2.000 # gpt-5 mini pricing as of October 2025

for doc in mongo_db.original_collection.find(limit=max_docs):
    text_id = doc["_id"]
    original_text = doc["text"]
    # print(original_text)
    print("Number of words:", len(original_text.split()))
    n_tokens_essay = num_tokens_from_string(string=original_text, encoding_name=ENCODING_NAME)
    n_tokens_prompt = num_tokens_from_string(string=CONFIG.PROMPT, encoding_name=ENCODING_NAME)
    print("Number of tokens for text essay:", n_tokens_essay)
    print("Total number of input tokens (+ prompt):", n_tokens_prompt + n_tokens_essay)
    total_price_usd = INPUT_PRICE_PER_ONE_M_TOKENS_USD * ((n_tokens_prompt + n_tokens_essay) / 1_000_000) + OUTPUT_PRICE_PER_ONE_M_TOKENS_USD * (n_tokens_essay * 3 / 1_000_000)
    print("Price for input and output tokens (USD):", total_price_usd)
    print("Price for input and output tokens (Euro):", total_price_usd * 0.86)


Number of words: 705
Number of tokens for text essay: 834
Total number of input tokens (+ prompt): 887
Price for input and output tokens (USD): 0.0052257499999999995
Price for input and output tokens (Euro): 0.004494145
